# Diabetes Risk Analysis and Predictive Modeling

Client-style portfolio notebook covering data cleaning, EDA, visualization, predictive modeling, explainability, and business recommendations.

## 1. Business Understanding

Objective: identify which patient characteristics are most associated with diabetes and build a practical model to estimate risk from biometric indicators.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid", context="notebook")
PROJECT_DIR = Path("..").resolve()
DATA_PATH = PROJECT_DIR / "data" / "diabetes.csv"
CHART_DIR = PROJECT_DIR / "assets" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
TARGET = "Outcome"
MEDICAL_ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

## 2. Dataset Overview

The dataset is the Pima Indians Diabetes Database. It contains 768 patient records, 8 predictor features, and a binary target `Outcome`.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
quality = {
    "duplicates": df.duplicated().sum(),
    "missing_values": df.isna().sum(),
    "invalid_zeros": df[MEDICAL_ZERO_AS_MISSING].eq(0).sum(),
    "target_distribution": df[TARGET].value_counts().sort_index()
}
quality

## 3. Data Cleaning and Preparation

Zero values in Glucose, BloodPressure, SkinThickness, Insulin, and BMI are not clinically valid, so they are treated as missing values and imputed with the median. Outliers are capped using the IQR rule.

In [ ]:
clean = df.copy()
clean[MEDICAL_ZERO_AS_MISSING] = clean[MEDICAL_ZERO_AS_MISSING].replace(0, np.nan)
clean[MEDICAL_ZERO_AS_MISSING] = SimpleImputer(strategy="median").fit_transform(clean[MEDICAL_ZERO_AS_MISSING])

outlier_summary = {}
for col in FEATURES:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_summary[col] = int(((clean[col] < lower) | (clean[col] > upper)).sum())
    clean[col] = clean[col].clip(lower, upper)

clean["BMI_Category"] = pd.cut(clean["BMI"], [0, 18.5, 25, 30, np.inf], labels=["Underweight", "Normal", "Overweight", "Obese"], include_lowest=True)
clean["Age_Group"] = pd.cut(clean["Age"], [20, 30, 40, 50, 60, 100], labels=["21-30", "31-40", "41-50", "51-60", "60+"], include_lowest=True)
clean["Glucose_Risk_Group"] = pd.cut(clean["Glucose"], [0, 99, 125, np.inf], labels=["Normal", "Elevated", "High"], include_lowest=True)
outlier_summary

## 4. Exploratory Data Analysis

In [ ]:
clean[FEATURES].agg(["mean", "median", "std", "min", "max"]).T.round(2)

In [ ]:
clean.groupby(TARGET)[["Glucose", "BMI", "Age", "BloodPressure", "Insulin"]].mean().rename(index={0: "No Diabetes", 1: "Diabetes"}).round(2)

In [ ]:
clean[FEATURES + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(ascending=False).round(3)

## 5. Professional Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), ["Glucose", "BMI", "Age", "BloodPressure"]):
    sns.histplot(data=clean, x=col, hue=TARGET, kde=True, bins=25, ax=ax, palette=["#2A9D8F", "#E76F51"])
    ax.set_title(f"{col} Distribution by Diabetes Outcome")
plt.tight_layout()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(clean[FEATURES + [TARGET]].corr(), annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlation Heatmap: Diabetes Risk Indicators")
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=clean, x="Glucose", y="BMI", hue=TARGET, size="Age", alpha=0.72, palette=["#2A9D8F", "#E76F51"])
plt.title("Glucose vs BMI, Sized by Age")
plt.tight_layout()

In [ ]:
sns.pairplot(clean[["Glucose", "BMI", "Age", "BloodPressure", TARGET]], hue=TARGET, palette=["#2A9D8F", "#E76F51"], diag_kind="hist", corner=True)

## 6. Predictive Modeling

Three models are compared: Logistic Regression, Random Forest, and Gradient Boosting. XGBoost can be added in a production environment if available, but Gradient Boosting is used here as a reliable boosting baseline.

In [ ]:
X = clean[FEATURES]
y = clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scaler_preprocessor = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), FEATURES)])
tree_preprocessor = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), FEATURES)])

model_specs = {
    "Logistic Regression": (Pipeline([("prep", scaler_preprocessor), ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]), {"model__C": [0.1, 1.0, 3.0]}),
    "Random Forest": (Pipeline([("prep", tree_preprocessor), ("model", RandomForestClassifier(random_state=RANDOM_STATE))]), {"model__n_estimators": [150, 250], "model__max_depth": [3, 5, None], "model__min_samples_leaf": [2, 5]}),
    "Gradient Boosting": (Pipeline([("prep", tree_preprocessor), ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))]), {"model__n_estimators": [80, 120], "model__learning_rate": [0.03, 0.06, 0.1], "model__max_depth": [2, 3]}),
}

fitted_models, rows, roc_data = {}, [], {}
for name, (pipeline, grid) in model_specs.items():
    search = GridSearchCV(pipeline, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    best = search.best_estimator_
    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]
    fitted_models[name] = best
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_test, y_proba))
    rows.append({
        "Model": name,
        "CV_ROC_AUC": search.best_score_,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Best_Params": search.best_params_
    })

metrics = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
metrics.round(3)

## 7. Explainability

In [ ]:
best_model_name = metrics.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
model_step = best_model.named_steps["model"]
if hasattr(model_step, "feature_importances_"):
    importance_values = model_step.feature_importances_
else:
    importance_values = np.abs(model_step.coef_[0])

feature_importance = pd.DataFrame({"Feature": FEATURES, "Importance": importance_values})
feature_importance["Importance"] = feature_importance["Importance"] / feature_importance["Importance"].sum()
feature_importance.sort_values("Importance", ascending=False).round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance.sort_values("Importance"), x="Importance", y="Feature", color="#264653")
plt.title(f"Feature Importance: {best_model_name}")
plt.tight_layout()

## 8. Business Insights

- Glucose is the clearest diabetes risk signal.
- BMI and Age provide practical segmentation dimensions for screening and dashboards.
- Predictive modeling can prioritize follow-up, but model results should not be framed as clinical diagnosis.
- Recommended dashboard KPIs: diabetes rate, average glucose, average BMI, high-risk glucose count, and patient count by age/BMI segment.